# Лабораторная 08. Data skew

Цель: увидеть, как один слишком частый ключ делает одну task намного тяжелее остальных.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-08-skew').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/07 11:53:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://0a370e2ebe67:4040


## Создаём перекошенный DataFrame
80% строк имеют `key = 'hot_key'`, остальные распределены по многим ключам.

In [2]:
n = 400_000
skewed = (
    spark.range(n)
    .withColumn('key', F.when(F.col('id') < n * 0.8, F.lit('hot_key')).otherwise(F.concat(F.lit('key_'), (F.col('id') % 1000))))
    .withColumn('value', F.rand(7))
    .repartition(8)
)
skewed.groupBy('key').count().orderBy(F.desc('count')).show(10)

+-------+------+
|    key| count|
+-------+------+
|hot_key|320000|
|key_703|    80|
|key_370|    80|
|key_238|    80|
|key_544|    80|
|key_285|    80|
|key_726|    80|
|key_407|    80|
|key_136|    80|
|key_617|    80|
+-------+------+
only showing top 10 rows



## groupBy по skewed key
Запустите aggregation и откройте Spark UI -> Stages. Ищите task duration: одна task может быть заметно дольше.

In [3]:
result = skewed.groupBy('key').agg(F.count('*').alias('cnt'), F.sum('value').alias('sum_value'))
result.explain('formatted')
result.count()

== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- Exchange (3)
         +- * Project (2)
            +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 400000, step=1, splits=Some(4))

(2) Project [codegen id : 1]
Output [2]: [CASE WHEN (cast(id#0L as double) < 320000.0) THEN hot_key ELSE concat(key_, cast((id#0L % 1000) as string)) END AS key#2, rand(7) AS value#5]
Input [1]: [id#0L]

(3) Exchange
Input [2]: [key#2, value#5]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=66]

(4) HashAggregate [codegen id : 2]
Input [2]: [key#2, value#5]
Keys [1]: [key#2]
Functions [2]: [partial_count(1), partial_sum(value#5)]
Aggregate Attributes [2]: [count#37L, sum#38]
Results [3]: [key#2, count#39L, sum#40]

(5) Exchange
Input [3]: [key#2, count#39L, sum#40]
Arguments: hashpartitioning(key#2, 8), ENSURE_REQUIREMENTS, [plan_id=70]

(6) HashAggregate [codegen id : 3]
Input [3]: [key#2, count#39L, sum#40]
K

1001

выполнение длилось 8 сек, всего 4 stage и первая самая длинная - 4s

## Skewed join
Join по перекошенному ключу часто ещё заметнее, потому что тяжёлая reduce partition должна сопоставить много строк.

In [4]:
dim = spark.createDataFrame([(f'key_{i}', f'name_{i}') for i in range(1000)] + [('hot_key', 'very_hot')], ['key', 'name'])
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
joined = skewed.join(dim, 'key')
joined.explain('formatted')
joined.count()

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (6)
   :  +- Exchange (5)
   :     +- Exchange (4)
   :        +- * Filter (3)
   :           +- * Project (2)
   :              +- * Range (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * Scan ExistingRDD (7)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 400000, step=1, splits=Some(4))

(2) Project [codegen id : 1]
Output [3]: [id#0L, CASE WHEN (cast(id#0L as double) < 320000.0) THEN hot_key ELSE concat(key_, cast((id#0L % 1000) as string)) END AS key#2, rand(7) AS value#5]
Input [1]: [id#0L]

(3) Filter [codegen id : 1]
Input [3]: [id#0L, key#2, value#5]
Condition : isnotnull(key#2)

(4) Exchange
Input [3]: [id#0L, key#2, value#5]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=187]

(5) Exchange
Input [3]: [id#0L, key#2, value#5]
Arguments: hashpartitioning(key#2, 8), ENSURE_REQUIREMENTS, [plan_id=188]

(6) Sort [codegen id : 2

26/07/07 16:10:39 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from 0a370e2ebe67:42579 in 10000 milliseconds
                                                                                

400000

я вижу в стадии join таски с разной длительностью - от 0.2 s (8667 записей) до 8 s (330450 записей) . 

Вопросы:

- Почему один key создаёт проблему? потому что все строки с одинаковым ключом гарантированно попадают в одну и ту же результирующую партицию и, соответственно, в одну task на стадии Reduce
- Почему один task работает дольше? Эта task работает дольше, потому что ей нужно обработать значительно больший объём данных
- Что видно в Task Duration и Shuffle Read? Task Duration варьирует от десятых секунд до десятка секунд в одном stage, число записей отличается в десятки раз)
- Почему простое увеличение памяти не всегда решает skew? потому что проблемная task всё равно выполняется в одном executor с ограничением на число ядер
- Какие способы борьбы возможны: AQE skew join, salting, изменение ключа, предварительная агрегация, broadcast маленькой стороны? все способы подходят, кроме предварительной агрегации - он не убирает skew

In [5]:
spark.stop()